# F02-P2 Nature

**Site Characterization: Nature, components 2.1 and 2.2.**

Characterizes the ecological quality of the AOI forest and habitat. Where `F02-P2 General`
answers "where is this site and what has happened to it", Nature answers "how good is what is
left, and does it matter for biodiversity".

> **Not runnable yet.** All analysis logic below is real Python. Only file access is stubbed.

**Status.** Complete as scoped: 2.1 FLII and 2.2 KBA.

**Known limit of that scope.** Both components need forest or a designated site to say anything. 2.1
FLII is a property of forest and returns not applicable without it, and 2.2 KBA returns a negative
sentence on most sites. A degraded grassland or a cropland targeted for planting therefore receives
almost no ecological description from this module, which is the site type where a RESTORE decision
most depends on the starting condition. Related asymmetry: the reference ecosystem has five classes
in the backend, but the only ecosystem quality metric here measures forest, so savanna and grassland
have no equivalent of FLII. Recorded as a scope limit, not as pending work.

## Handoff

Reads nothing from `F02-P2 General` at present. Both Nature components derive their own forest
extent from `forest_mask_2024`, so this notebook can run independently. It writes
`outputs/<aoi_id>__F02-P2-nature.json`.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np

from config import *
from common import *

In [2]:
aoi_id = AOI_ID   # AOI is set in config.py (AOI_PATH, AOI_ID); change it there, then restart the kernel

aoi = prepare_aoi(gpd.read_file(AOI_PATH))
print(f"AOI {aoi_id}: {fmt_ha(aoi.area_ha)}")

results: dict[str, ComponentResult] = {}

AOI aoi1: 67,439 ha


---
## 2.1 Forest Landscape Integrity (FLII)

Reports the landscape integrity of the AOI forest as a headline mean score out of 10, with a
High / Medium / Low breakdown in the narrative. Integrity is the degree to which a forest is
still intact, connected, and free of human pressure.

**Data.** `flii_forest_mosaic_SEA_300m.tif` (continuous 0 to 10, on forest) and
`flii_class_mosaic_SEA_300m.tif` (1 = Low, 2 = Medium, 3 = High, on forest). Based on the Forest
Landscape Integrity Index (Grantham et al. 2020, Nat. Commun. 11:5978), reimplemented and
calibrated on the SEA data stack. Landscape scale, native 300 m.

**Calibration warning.** The 0 to 10 values are calibrated on the pooled SEA distribution, so
they are not one to one with the published global FLII product. Present them as the SEA forest
integrity layer, internally consistent within this run. Do not claim absolute global integrity.
Class breaks follow the paper: High at or above 9.6, Low at or below 6.0, Medium in between.

**Decisions locked.**

- FLII is a property of forest, so the summary covers AOI forest area only. Denominator = AOI
  forest area, the same forest used in 1.5 and 1.6.
- Headline is the mean FLII score out of 10, a single big number for the frontend.
- The narrative reports the High, Medium and Low share and names the predominant class.

**Example render (predominantly High).**

> **Forest landscape integrity: 8.8 / 10**
>
> Of the forest in this area, 68% has high landscape integrity, 24% medium, and 8% low. The
> forest is predominantly high integrity, indicating largely intact and well-connected forest
> under low human pressure.

**Downstream use.** FLII is a biodiversity and ecosystem quality proxy feeding Triple Win
Pillar 1, and a pathway signal: high integrity favours PROTECT, low integrity favours RESTORE or
MANAGE. It also underpins the SCeNe high-integrity NbS criteria.

In [3]:
FLII_LOW, FLII_MEDIUM, FLII_HIGH = 1, 2, 3

FLII_GLOSS = {
    FLII_HIGH: "indicating largely intact and well-connected forest under low human pressure",
    FLII_MEDIUM: (
        "indicating moderately modified forest with some fragmentation or human pressure"
    ),
    FLII_LOW: "indicating heavily modified and fragmented forest under high human pressure",
}


def analyze_flii(aoi: AOI) -> ComponentResult:
    """Component 2.1. Forest landscape integrity over the AOI forest."""
    # The FLII rasters are already masked to forest upstream, so their valid extent defines the
    # forest here. forest_mask_2024 is loaded only to catch the "no forest at all" case early,
    # so the message matches 1.5 and 1.6 rather than saying "no FLII data".
    if forest_mask_2024(aoi).is_empty:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "No forest is present in this project area, so landscape integrity cannot be "
            "assessed.",
        )

    score = load_raster_clipped(FLII_FOREST_RASTER, aoi, resampling="bilinear")
    classes = load_raster_clipped(FLII_CLASS_RASTER, aoi, resampling="nearest")

    forest_area_ha = classes.valid_area_ha
    if forest_area_ha <= 0 or score.valid_count == 0:
        return not_applicable(
            "2.1 Forest Landscape Integrity",
            "The forest integrity layer does not cover the forest in this project area.",
        )

    mean_flii = float(np.ma.mean(score.values))  # 0 to 10, one decimal on display

    rows = tabulate_classes(classes, FLII_CLASSES, denominator_ha=forest_area_ha)
    by_code = {r.code: r for r in rows}
    dom = dominant(rows)

    narrative = sentences(
        f"Of the forest in this area, {fmt_pct(by_code[FLII_HIGH].pct)} has high landscape "
        f"integrity, {fmt_pct(by_code[FLII_MEDIUM].pct)} medium, and "
        f"{fmt_pct(by_code[FLII_LOW].pct)} low.",
        f"The forest is predominantly {dom.label.lower()} integrity, {FLII_GLOSS[dom.code]}.",
    )

    return ComponentResult(
        component="2.1 Forest Landscape Integrity",
        applicable=True,
        narrative=narrative,
        tables={"integrity": rows},
        rasters={"2.1_flii_score": score, "2.1_flii_class": classes},
        values={
            "mean_flii": mean_flii,  # headline big number
            "dominant_class": dom.code,
            "pct_high": by_code[FLII_HIGH].pct,
            "forest_area_ha": forest_area_ha,
        },
    )


results["2.1"] = analyze_flii(aoi)
show_result(results["2.1"])

[2.1 Forest Landscape Integrity]
  Of the forest in this area, 0% has high landscape integrity, 45% medium, and 55% low. The forest is predominantly low integrity, indicating heavily modified and fragmented forest under high human pressure.
  integrity:


,code,label,area_ha,pct
0,1,Low,22259.439455,55.058366
1,2,Medium,18169.365774,44.941634
2,3,High,0.000000,0.000000


{'mean_flii': 2.782495220954318,
 'dominant_class': 1,
 'pct_high': 0.0,
 'forest_area_ha': 40428.805228613004}

---
## 2.2 Key Biodiversity Areas (KBA)

Reports whether the AOI overlaps Key Biodiversity Areas, with overlap area, share and a
narrative.

**A KBA is not a protected area.** It is a site that contributes significantly to the global
persistence of biodiversity (IUCN KBA Standard 2016). It may or may not be legally protected.
This is a different lens from 1.3 (WDPA, legal status) and the two complement each other. The
narrative emphasises biodiversity importance, not protection.

**Data.** `KBA_polygon.shp`, World Database of Key Biodiversity Areas, BirdLife International
and the KBA Partnership.

**Decisions locked.**

- Mirrors 1.3 mechanics: headline overlap = union of KBA polygons, so overlapping or nested
  sites are not double counted. No sliver threshold, because any KBA overlap is material.
- Denominator = total AOI area. A KBA concerns the whole site, not only its forest.
- The narrative gives the KBA name only, no criteria or type.

**To verify against the actual file.** Name field assumed here: `IntName`.

**Example render.**

> This project area overlaps 210 ha (17%) of a Key Biodiversity Area, Bukit Tigapuluh. Key
> Biodiversity Areas are sites that contribute significantly to the global persistence of
> biodiversity.

**Downstream use.** Feeds Triple Win Pillar 1 and acts as a safeguard and eligibility signal. A
KBA that is not also under WDPA protection (1.3) is a biodiversity important but unprotected
site, which is a strong PROTECT and additionality rationale.

In [4]:
KBA_DEFINITION = (
    "Key Biodiversity Areas are sites that contribute significantly to the global persistence "
    "of biodiversity."
)


@dataclass(frozen=True)
class KbaSite:
    name: str
    area_ha: float


def analyze_kba(aoi: AOI) -> ComponentResult:
    """Component 2.2. Overlap with Key Biodiversity Areas."""
    gdf = load_vector_intersecting(KBA_POLYGON, aoi)

    if gdf.empty:
        return ComponentResult(
            component="2.2 Key Biodiversity Areas",
            applicable=True,  # "no overlap" is a real answer, not a missing one
            narrative="This project area does not overlap any Key Biodiversity Areas.",
            tables={"sites": []},
            values={"kba_ha": 0.0, "kba_pct": 0.0, "kba_site_count": 0},
        )

    kba_ha = union_overlap_ha(aoi, gdf)
    kba_pct = safe_pct(kba_ha, aoi.area_ha)

    areas = per_feature_overlap_ha(aoi, gdf)
    sites = sort_by_area([
        KbaSite(name=str(gdf.iloc[i].get("IntName", "Unnamed KBA")), area_ha=float(areas[i]))
        for i in range(len(gdf))
    ])

    if len(sites) == 1:
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of a Key "
            f"Biodiversity Area, {sites[0].name}."
        )
    else:
        largest, *rest = sites
        rest_text = oxford_join(f"{s.name} ({fmt_ha(s.area_ha)})" for s in rest)
        head = (
            f"This project area overlaps {fmt_ha(kba_ha)} ({fmt_pct(kba_pct)}) of Key "
            f"Biodiversity Areas, across {len(sites)} sites. The largest is {largest.name} "
            f"({fmt_ha(largest.area_ha)}), followed by {rest_text}."
        )

    return ComponentResult(
        component="2.2 Key Biodiversity Areas",
        applicable=True,
        narrative=sentences(head, KBA_DEFINITION),
        tables={"sites": sites},
        values={"kba_ha": kba_ha, "kba_pct": kba_pct, "kba_site_count": len(sites)},
    )


results["2.2"] = analyze_kba(aoi)
show_result(results["2.2"])

[2.2 Key Biodiversity Areas]
  This project area overlaps 14,317 ha (21%) of a Key Biodiversity Area, Gunung Niut-Poteng. Key Biodiversity Areas are sites that contribute significantly to the global persistence of biodiversity.
  sites:


,name,area_ha
0,Gunung Niut-Poteng,14316.718525


{'kba_ha': 14316.718524669994,
 'kba_pct': 21.22913156918525,
 'kba_site_count': 1}

---
## 2.3 Habitat Area

## AOH is a refinement of IUCN Range

The **AOH approach** refines coarse species range maps into spatially-explicit estimates of available suitable habitat, following the globally standardized methods of *Brooks et al., 2019*

For each species (~6,000 in total), AOH generated through the following sequential filtering steps:

1. Retain only polygons classified as **extant and native** in the IUCN/BirdLife dataset and remove polygons classified as introduced, uncertain, or extinct.

2. Extract habitat preferences from **IUCN Red List species assessment** and mask the range map to include only pixels corresponding to suitable habitat classes based on the crosswalk.

3. Apply species-specific elevational limits using a high-resolution digital elevation model. Where relevant, additional climatic constraints will be applied to refine habitat suitability for habitat types, such as savannas.

---

The resulting raster represents the intersection of **range, habitat, and environmental suitability**. Outputs will be binary (**suitable/unsuitable**) and stored as species-level rasters.

The accuracy and validity of AOH assessed using species occurrence data from the **Global Biodiversity Information Facility (GBIF)**, as well as data collected from three NbS pilot sites. Validation assess the proportion of independent occurrence records that fall within mapped AOH, providing an indication of model performance

---

## Example render

> The selected area is a suitable habitat for a wide range of wildlife, including

| Species summary       | Number of species |
| --------------------- | ----------------: |
| **Masked species**    |            **5** |
| **Reference species** |           **264** |

### Species and IUCN Status

| Species Name              | IUCN Status |
| ------------------------- | :---------: |
| `Erythropitta_venusta`    |    **CR**   |
| `Cochoa_beccarii`         |    **EN**   |
| `Rhyticeros_undulatus`    |    **EN**   |
| `Cyornis_caerulatus`      |    **VU**   |
| `Rubigula_dispar`         |    **VU**   |



In [ ]:
# ================FOR PRE-FEASIBILITY DOCUMENT================================
# =============================================================================
# Script 2 of 2 - Add IUCN status to the masked species list (VLOOKUP)
# The 2nd script is used to retrieved data for Document template
#
# Inputs:
#   - sea_birds_inventory.xlsx        masked result: species inside the AOI
#   - sea_birds_inventory_DUMMY.xlsx  reference: species + IUCN status
#
# Process:
#   For each species in the masked list, look up its 'iucn_status' in the
#   DUMMY reference table (the pandas equivalent of a VLOOKUP) and write the
#   masked species list with the status attached.
# =============================================================================

from pathlib import Path

import pandas as pd


# =============================================================================
# Configuration
# =============================================================================

# Masked result: list of species found inside the user polygon (left table).
masked_excel_path = (
    r"\\OPENMEDIAVAULT\geospatial\NBS\AoH\temp_pilot\sea_birds_inventory.xlsx"
)

# Reference table: all species with their IUCN status (lookup source).
dummy_excel_path = (
    r"\\OPENMEDIAVAULT\geospatial\NBS\AoH\temp_pilot\sea_birds_inventory_DUMMY.xlsx"
)

# Final output: masked species list with IUCN status.
output_excel_path = (
    r"\\OPENMEDIAVAULT\geospatial\NBS\AoH\temp_pilot\sea_birds_inventory_iucn.xlsx"
)

# Sheet to read from each workbook (0 = first sheet).
MASKED_SHEET = 0
DUMMY_SHEET = 0

# Species-name column in each file (the VLOOKUP key).
MASKED_SPECIES_COLUMN = "species"
DUMMY_SPECIES_COLUMN = "species"

# IUCN status column in the DUMMY reference table.
IUCN_COLUMN = "iucn_status"

# Label used when a masked species is not present in the reference table.
UNMATCHED_LABEL = "Not in reference"

# IUCN categories from most to least threatened, for a tidy summary order.
IUCN_ORDER = ["CR", "EN", "VU", "NT", "LC", "DD", "NE"]


# =============================================================================
# Helpers
# =============================================================================

def normalize_name(value):
    """Normalize a species name for tolerant matching (case/underscore/spacing)."""
    if value is None:
        return ""
    text = str(value).strip().lower()
    text = text.replace("_", " ")
    return " ".join(text.split())


def require_column(table, column, source):
    if column not in table.columns:
        raise KeyError(
            f"Column '{column}' not found in {source}. "
            f"Available columns: {list(table.columns)}"
        )


# =============================================================================
# Load inputs
# =============================================================================

masked = pd.read_excel(masked_excel_path, sheet_name=MASKED_SHEET)
require_column(masked, MASKED_SPECIES_COLUMN, "the masked species file")

dummy = pd.read_excel(dummy_excel_path, sheet_name=DUMMY_SHEET)
require_column(dummy, DUMMY_SPECIES_COLUMN, "the DUMMY reference file")
require_column(dummy, IUCN_COLUMN, "the DUMMY reference file")

print(f"Masked species: {len(masked)}")
print(f"Reference species: {len(dummy)}")


# =============================================================================
# VLOOKUP - pull iucn_status from DUMMY into the masked list
# =============================================================================

# Build a normalized lookup: species key -> IUCN status.
status_by_key = {}
for _, row in dummy.iterrows():
    key = normalize_name(row[DUMMY_SPECIES_COLUMN])
    if not key:
        continue
    status = row[IUCN_COLUMN]
    status_by_key[key] = None if pd.isna(status) else str(status).strip()

# Attach status to each masked species.
result = masked.copy()
keys = result[MASKED_SPECIES_COLUMN].map(normalize_name)

result["matched"] = keys.isin(status_by_key)
result[IUCN_COLUMN] = keys.map(status_by_key)
result[IUCN_COLUMN] = result[IUCN_COLUMN].where(
    result[IUCN_COLUMN].notna(), UNMATCHED_LABEL
)


# =============================================================================
# Sort by IUCN severity, then by species name
# =============================================================================

order_rank = {status: rank for rank, status in enumerate(IUCN_ORDER)}
result["_rank"] = result[IUCN_COLUMN].map(
    lambda status: order_rank.get(status, len(IUCN_ORDER) + 1)
)
result = (
    result.sort_values(["_rank", MASKED_SPECIES_COLUMN])
    .drop(columns="_rank")
    .reset_index(drop=True)
)


# =============================================================================
# Write the result
# =============================================================================

output_path = Path(output_excel_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="masked_species_iucn", index=False)

    worksheet = writer.sheets["masked_species_iucn"]
    for column_cells in worksheet.columns:
        longest = max(
            (len(str(cell.value)) for cell in column_cells if cell.value is not None),
            default=0,
        )
        letter = column_cells[0].column_letter
        worksheet.column_dimensions[letter].width = min(longest + 2, 60)


# =============================================================================
# Display output
# =============================================================================

print()
print("By IUCN status:")
for status, count in result[IUCN_COLUMN].value_counts().items():
    print(f"  {status}: {count}")

unmatched = result.loc[~result["matched"], MASKED_SPECIES_COLUMN].tolist()
if unmatched:
    print()
    print(f"Warning: {len(unmatched)} species not found in the reference table:")
    for name in unmatched:
        print(f"  - {name}")

print()
print(f"Output written to: {output_path}")

---
## 2.5 Key Species Presence

Reports the species occurence inside the AOI, with overlap point, share the quantitative data

The Key Species Presence is occurence of keystone or flagship species from different region accross Southeast-Asia. The data comes from GBIF (Global Biodiversity Information Facility) by compiling sources from Natural History Museums, Citizen Science Networks, Academic & Research Institutions, Government & Conservation Agencies. Following the data standards under The Darwin Core Standard (DwC), Ecological Metadata Language, and The Biological Collection Access Service.

---

The resulting shapefile represents the intersection of **species, individualCount, eventDate and basisofRecords**. Main outputs will be the total occurence (**Species Name and Record Count**)

---

## Example render

> The selected area recorded observations of **2** species.

| Occurrence summary    | Value |
| --------------------- | ----: |
| **Unique species**    | **2** |
| **Total records**     |     4 |
| **Total individuals** |     6 |

### Species Occurrence

| Species Name               | Total Occurrence | Record Count | Latest Encounter | Basis of Record   |
| -------------------------- | ---------------: | -----------: | ---------------- | ----------------- |
| `Buceros rhinoceros`       |            **6** |            3 | 23 February 2017 | HUMAN_OBSERVATION |
| `Symphalangus syndactylus` |            **0** |            1 | 03 November 2014 | HUMAN_OBSERVATION |



In [ ]:
# =============================================================================
# Species Occurrence Analysis (local shapefile)
#
# Input:
#   1. User-drawn polygon (vector file: .shp / .geojson / .gpkg)
#   2. Species occurrence point layer (shapefile)
#
# Process:
#   1-2. Keep occurrence points that fall inside the user polygon.
#   3.   Group records by species.
#   4.   Per species: sum individualCount, latest eventDate, modal basisOfRecord.
#   5.   Count total unique species.
#
# Output:
#   - Total unique species
#   - Species name, total occurrence, latest encounter (DD Month YYYY),
#     most frequent basis of record
#
# Note: shapefile field names are truncated to 10 characters, so Darwin Core
# fields are often stored as 'individual', 'basisOfRec', etc. The column
# resolver below handles that automatically.
# =============================================================================
# =============================================================================
# Configuration
# =============================================================================

# ---- Species occurrence points -----------------------------------------------
species_points_path = r"\\OPENMEDIAVAULT\geospatial\NBSTOOLV3\BIODIVERSITY\GBIF\key_species.shp"

# ---- User polygon ------------------------------------------------------------
user_polygon_path = r"Z:\NbS_Tools\Dummy\AOI_4326.shp"

# ---- Output ------------------------------------------------------------------
output_excel_path = (
    r"\\OPENMEDIAVAULT\geospatial\NBS\AoH\temp_pilot\species_occurrence_summary.xlsx"
)

# ---- Desired attribute columns (Darwin Core names) ---------------------------
# The resolver matches these against the (possibly truncated) shapefile fields.
SPECIES_COLUMN = "species"
COUNT_COLUMN = "individualCount"
DATE_COLUMN = "eventDate"
BASIS_COLUMN = "basisOfRecord"

# Spatial predicate: "intersects" includes points on the boundary,
# "within" is strictly interior.
SPATIAL_PREDICATE = "intersects"


# =============================================================================
# Helpers
# =============================================================================

def resolve_column(gdf, desired):
    """Find the actual column matching a desired name, allowing for
    shapefile 10-char truncation and case differences."""
    columns = list(gdf.columns)

    # 1. Exact match.
    if desired in columns:
        return desired

    lower_map = {col.lower(): col for col in columns}

    # 2. Case-insensitive exact match.
    if desired.lower() in lower_map:
        return lower_map[desired.lower()]

    # 3. Truncation: the shapefile field is a prefix of the desired name.
    #    Pick the longest such prefix match.
    candidates = [
        col for col in columns
        if desired.lower().startswith(col.lower()) and len(col) >= 4
    ]
    if candidates:
        return max(candidates, key=len)

    raise KeyError(
        f"Could not find a column for '{desired}'. "
        f"Available columns: {columns}"
    )


def most_frequent(series):
    modes = series.dropna().mode()
    return modes.iat[0] if not modes.empty else None


# =============================================================================
# Load inputs
# =============================================================================

points = gpd.read_file(species_points_path)
if points.empty:
    raise ValueError("The species occurrence layer contains no features.")

polygon = gpd.read_file(user_polygon_path)
if polygon.empty:
    raise ValueError("The user polygon contains no features.")

print(f"Loaded {len(points)} occurrence points.")

# Resolve the attribute columns against the actual (truncated) field names.
species_col = resolve_column(points, SPECIES_COLUMN)
count_col = resolve_column(points, COUNT_COLUMN)
date_col = resolve_column(points, DATE_COLUMN)
basis_col = resolve_column(points, BASIS_COLUMN)

print("Resolved columns:")
print(f"  species        -> {species_col}")
print(f"  individualCount-> {count_col}")
print(f"  eventDate      -> {date_col}")
print(f"  basisOfRecord  -> {basis_col}")


# =============================================================================
# Steps 1-2 - Keep points inside the polygon
# =============================================================================

# Align CRS: reproject the polygon to the points CRS.
if points.crs is None:
    raise ValueError("The occurrence layer has no CRS.")

if polygon.crs is None:
    print("Warning: polygon has no CRS; assuming it matches the points layer.")
    polygon = polygon.set_crs(points.crs)
elif polygon.crs != points.crs:
    polygon = polygon.to_crs(points.crs)

# Dissolve the polygon to a single geometry so a point can't match more than
# one polygon feature (which would double-count it).
try:
    merged_geometry = polygon.geometry.union_all()
except AttributeError:  # older geopandas
    merged_geometry = polygon.geometry.unary_union

polygon_single = gpd.GeoDataFrame(geometry=[merged_geometry], crs=polygon.crs)

clipped_points = gpd.sjoin(
    points,
    polygon_single,
    predicate=SPATIAL_PREDICATE,
    how="inner",
).drop(columns="index_right")

print(f"Occurrence points inside the polygon: {len(clipped_points)}")

if clipped_points.empty:
    raise SystemExit("No occurrence points fall inside the polygon.")


# =============================================================================
# Step 3-4 - Clean, group by species, aggregate
# =============================================================================

# individualCount -> numeric (nulls become NaN and are ignored by sum).
clipped_points[count_col] = pd.to_numeric(
    clipped_points[count_col], errors="coerce"
)

# eventDate -> datetime so "max" is chronological, not lexical.
clipped_points[date_col] = pd.to_datetime(
    clipped_points[date_col], errors="coerce"
)

# Records with no species name cannot be grouped meaningfully.
clipped_points = clipped_points.dropna(subset=[species_col])

result = (
    clipped_points
    .groupby(species_col)
    .agg(
        total_occurrence=(count_col, "sum"),      # summed individualCount
        record_count=(species_col, "count"),      # number of occurrence rows
        latest_encounter=(date_col, "max"),
        basis_of_record=(basis_col, most_frequent),
    )
    .reset_index()
    .rename(columns={species_col: "species"})
)


# =============================================================================
# Step 5 - Total unique species
# =============================================================================

total_unique_species = result["species"].nunique()


# =============================================================================
# Format output
# =============================================================================

result["total_occurrence"] = result["total_occurrence"].round().astype("Int64")
result["record_count"] = result["record_count"].astype("Int64")

result["latest_encounter"] = result["latest_encounter"].apply(
    lambda value: value.strftime("%d %B %Y") if pd.notna(value) else None
)

result = result.sort_values(
    "total_occurrence", ascending=False
).reset_index(drop=True)


# =============================================================================
# Write and display
# =============================================================================

output_path = Path(output_excel_path)
output_path.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    result.to_excel(writer, sheet_name="species_summary", index=False)

    worksheet = writer.sheets["species_summary"]
    for column_cells in worksheet.columns:
        longest = max(
            (len(str(cell.value)) for cell in column_cells if cell.value is not None),
            default=0,
        )
        letter = column_cells[0].column_letter
        worksheet.column_dimensions[letter].width = min(longest + 2, 60)

# print()
# print(f"Total unique species: {total_unique_species}")
# print()
# print(result.to_string(index=False))
# print()
# print(f"Summary written to: {output_path}")

# ---- Notebook rendering ------------------------------------------------------
# In Jupyter / ArcGIS notebooks this shows the summary as a formatted table.
try:
    from IPython.display import display

    print()
    print(f"Total unique species: {total_unique_species}")
    display(result)
except ImportError:
    pass

# Leaving `result` as the final expression lets a notebook cell render it too.
result

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_NATURE)
print(f"Saved {path}")